<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [ ]:

# !pip3 install folium
# !pip3 install wget
# !pip3 install pandas


In [ ]:

import folium
import pandas as pd
# import wget


In [ ]:

# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon


If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [ ]:

# Download and read the `spacex_launch_geo.csv`
# spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
# spacex_df=pd.read_csv(spacex_csv_file)
path = 'your_path'
spacex_csv_file = path + 'spacex_launch_geo.csv'
spacex_df=pd.read_csv(spacex_csv_file)
spacex_df.head(1)


Now, you can take a look at what are the coordinates for each site.


In [ ]:

# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class', 'Date']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long', 'Date']]
launch_sites_df.head()


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [ ]:

# Start location is NASA Johnson Space Center

nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)


We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [ ]:

# Create a orange circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, 
                        radius=1000, 
                        color='black',
                        fill=True,
                        # fill_color='blue',
                        fill_opacity=0.6
                       ).add_child(folium.Popup('NASA Johnson Space Center')
                                  )

# Circle at NASA Johnson Space Center's with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:red;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)


and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`
- Task: Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map.
- An example of folium.Circle:
`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`
- An example of folium.Marker:
`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`

In [ ]:

# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label

sites_names = []
latitudes = []
longitudes = []

for n in list(launch_sites_df.index):
    sites_names.append(launch_sites_df.iloc[n,0])
    latitudes.append(launch_sites_df.iloc[n,1])
    longitudes.append(launch_sites_df.iloc[n,2]) 

print(sites_names)
print(latitudes)
print(longitudes)


In [ ]:

# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=3)

#`folium.Circle` and `folium.Marker` for each launch site on the site map
for latitude, longitude, site_name in zip(latitudes, longitudes, sites_names):
    circle = folium.Circle([latitude, longitude], radius=100, color='red', fill=True, fill_color='black',fill_opacity=0.4)
    marker = folium.map.Marker([latitude, longitude],icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0), 
                                html='<div style="font-size: 16; color: black;"><b>%s</b></div>' % site_name,))
    site_map.add_child(circle)
    site_map.add_child(marker)
site_map


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [ ]:

spacex_df.tail()


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [ ]:

# site_map = folium.Map(location=nasa_coordinate, zoom_start=2)
marker_cluster = MarkerCluster()


_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [ ]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red


In [ ]:

# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail()


_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [ ]:

# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']

# for index, row in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    
    # marker = folium.Marker(...)
    # marker_cluster.add_child(marker)

# site_map

for lat, lng, label, color, date, in zip(spacex_df.Lat, 
                                         spacex_df.Long, 
                                         spacex_df['class'], 
                                         spacex_df['marker_color'], 
                                         spacex_df['Date']):
    folium.Marker(
    location=[lat, lng],
    icon=folium.Icon(color='white', icon_color=color),
    popup=date,
    # popup=label,
    ).add_to(marker_cluster)
site_map


- For each row in "spacex_df", create a Marker object with its coordinate and customize the Marker's icon property
to indicate if this launch was successed or failed, e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [ ]:

# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map - OK!!!

formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [ ]:

from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    
    R = 6373.0    # approximate radius of earth in km

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance
    

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [ ]:

# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# launch sites nearest coastlines coordinates

# CCAFS LC-40 	[28.562302, -80.577356]    <---->    west coast [28.564, -80.568]
# CCAFS SLC-40 	[28.563197,	-80.576820]    <---->    west coast [28.564. -80.568] 
# KSC LC-39A 	[28.573255,	-80.646895]    <---->    west coast [28.639, -80.596]
# VAFB SLC-4E 	[34.632834,	-120.610745]   <---->    east coast [34.636, -12.624]

closest_coast_coordinates = {'Launch Site':['CCAFS_LC_40','CCAFS_SLC_40','KSC_LC_30','VAFB_SLC_4E'],
                             'Lat_Coast':[28.563, 28.564, 28.600, 34.636],
                             'Long_Coast':[-80.568, -80.568, -80.586, -120.625],
                             # 'Lat_Road':[28.56158, 28.56158, 28.576, 34.63273],
                             # 'Long_Road':[-80.57727, -80.57727, -80.645, -120.61019]
                            }

c3 = closest_coast_coordinates
c3_df = pd.DataFrame(c3)
c3_df

# launch sites distances to nearest coastlines


In [ ]:

launch_sites_c3_df = pd.concat([launch_sites_df, c3_df[['Lat_Coast','Long_Coast']]],axis=1)
launch_sites_c3_df = launch_sites_c3_df.drop(['Date'], axis=1)
launch_sites_c3_df


In [ ]:

# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

distance_coastline = []
for index, row in launch_sites_c3_df.iterrows():
    distance_coastline.append(calculate_distance(row.iloc[1],
                                                 row.iloc[2],
                                                 row.iloc[3],
                                                 row.iloc[4]))
    
distance_coastline = pd.DataFrame({'Distance_Coastline':distance_coastline})
launch_sites_c3_df = pd.concat([launch_sites_c3_df,distance_coastline],axis=1)
# print(distance_coastline, distance_road)
launch_sites_c3_df


_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [ ]:

# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )

for index, row in launch_sites_c3_df.iterrows():
    distance_marker = folium.Marker(
    [row['Lat_Coast'],row['Long_Coast']],
    icon=DivIcon(
    icon_size=(20,20),
    icon_anchor=(0,0),
    html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(row['Distance_Coastline']),))
    site_map.add_child(distance_marker)

site_map


In [ ]:

# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# lines=folium.PolyLine(locations=coordinates, weight=1)
# site_map.add_child(lines)

ln1 = folium.PolyLine(locations=[(28.562302, -80.577356),(28.563, -80.568)], weight=2, color="darkblue") 
ln1.add_child(folium.Popup("CCAFS LC-40 closest coastline"))

ln2 = folium.PolyLine(locations=[(28.563197, -80.576820), (28.564, -80.568)], weight=2, color="darkblue")
ln2.add_child(folium.Popup("CCAFS SLC-40 closest coastline"))

ln3 = folium.PolyLine(locations=[(28.573255, -80.6468959),(28.600, -80.586)], weight=2, color="darkblue")
ln3.add_child(folium.Popup("KSC LC-39A closest coastline"))

ln4 = folium.PolyLine(locations=[(34.632834, -120.610745), (34.636, -120.625)], weight=2, color="darkblue")
ln4.add_child(folium.Popup("VAFB SLC-4E closest coastline"))

ln1.add_to(site_map)
ln2.add_to(site_map)
ln3.add_to(site_map)
ln4.add_to(site_map)

site_map.add_child(ln1)
site_map.add_child(ln2)
site_map.add_child(ln3)
site_map.add_child(ln4)

site_map

_TODO:_ Similarly, you can draw a line between a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first

In [ ]:

# launch site to its closest city, railway, highway, etc.

# CCAFS LC-40 	[28.562302, -80.577356]    <---->    city [228.61300, -80.80520]
# CCAFS SLC-40 	[28.563197,	-80.576820]    <---->    city [228.61300, -80.80520] 
# KSC LC-39A 	[28.573255,	-80.646895]    <---->    city [228.61300, -80.80520]
# VAFB SLC-4E 	[34.632834,	-120.610745]   <---->    city [34.63921, -120.46465]

# CCAFS LC-40 	[28.562302, -80.577356]    <---->    highway [28.564, -80.568]
# CCAFS SLC-40 	[28.563197,	-80.576820]    <---->    highway [28.564. -80.568] 
# KSC LC-39A 	[28.573255,	-80.646895]    <---->    highway [28.639, -80.596]
# VAFB SLC-4E 	[34.632834,	-120.610745]   <---->    highway [34.636, -120.624]

# CCAFS LC-40 	[28.562302, -80.577356]    <---->    railway [28.56430, -80.58660]
# CCAFS SLC-40 	[28.563197,	-80.576820]    <---->    railway [28.56526, -80.58660] 
# KSC LC-39A 	[28.573255,	-80.646895]    <---->    railway [28.57267, -80.65392]
# VAFB SLC-4E 	[34.632834,	-120.610745]   <---->    railway [34.63503, -120.62428]

closest_various_coordinates = {'Launch Site':['CCAFS_LC_40','CCAFS_SLC_40','KSC_LC_30','VAFB_SLC_4E'],
                             'Lat_Coast':[28.563, 28.564, 28.600, 34.636],
                             'Long_Coast':[-80.568, -80.568, -80.586, -120.625],
                             'Lat_Road':[28.56158, 28.56158, 28.576, 34.63273],
                             'Long_Road':[-80.57727, -80.57727, -80.645, -120.61019],
                             'Lat_City':[28.6130, 28.6130, 28.6130, 34.63921],
                             'Long_City':[-80.8052, -80.8052, -80.8052, -120.46465], 
                             'Lat_Railway':[28.5643, 28.56526, 28.57267, 34.63503],
                             'Long_Railway':[-80.5866, -80.5866, -80.65392, -120.62428]                                
                            }

cvc = closest_various_coordinates
cvc_df = pd.DataFrame(cvc)
cvc_df


In [ ]:

# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site

ln5 = folium.PolyLine(locations=[(28.562302, -80.577356), (28.6130, -80.8052)], weight=2, color="darkgoldenrod")
ln5.add_child(folium.Popup("CCAFS LC-40 closest city"))

ln6 = folium.PolyLine(locations=[(28.563197, -80.576820), (28.6130, -80.8052)], weight=2, color="darkgoldenrod")
ln6.add_child(folium.Popup("CCAFS SLC-40 closest city"))

ln7 = folium.PolyLine(locations=[(28.573255, -80.6468959), (28.6130, -80.8052)], weight=2, color="darkgoldenrod")
ln7.add_child(folium.Popup("KSC LC-39A closest city"))

ln8 = folium.PolyLine(locations=[(34.632834, -120.610745), (34.63921, -120.46465)], weight=2, color="darkgoldenrod")
ln8.add_child(folium.Popup("VAFB SLC-4E closest city"))

ln9 = folium.PolyLine(locations=[(28.562302, -80.577356), (28.5643, -80.5866)], weight=2, color="darkred")
ln9.add_child(folium.Popup("CCAFS LC-40 closest railway"))

ln10 = folium.PolyLine(locations=[(28.563197, -80.576820), (28.56526, -80.5866)], weight=2, color="darkred")
ln10.add_child(folium.Popup("CCAFS SLC-40 closest railway"))

ln11 = folium.PolyLine(locations=[(28.573255, -80.6468959), (28.57267, -80.65392)], weight=2, color="darkred")
ln11.add_child(folium.Popup("KSC LC-39A closest railway"))

ln12 = folium.PolyLine(locations=[(34.632834, -120.610745), (34.63503, -120.62428)], weight=2, color="darkred")
ln12.add_child(folium.Popup("VAFB SLC-4E closest railway"))

ln13 = folium.PolyLine(locations=[(28.563197, -80.576820), (28.56158, -80.57727)], weight=2, color="darkgreen")
ln13.add_child(folium.Popup("CCAFS SLC-40 closest road"))

ln14 = folium.PolyLine(locations=[(28.573255, -80.6468959),(28.576, -80.645)], weight=2, color="darkgreen")
ln14.add_child(folium.Popup("KSC LC-39A closest road"))

ln15 = folium.PolyLine(locations=[(34.632834, -120.610745), (34.63273, -120.61019)], weight=2, color="darkgreen")
ln15.add_child(folium.Popup("VAFB SLC-4E closest road"))

ln16 = folium.PolyLine(locations=[(28.562302, -80.577356),(28.56158, -80.57727)], weight=2, color="darkgreen")
ln16.add_child(folium.Popup("CCAFS LC-40 closest road"))

site_map.add_child(ln5)
site_map.add_child(ln6)
site_map.add_child(ln7)
site_map.add_child(ln8)
site_map.add_child(ln9)
site_map.add_child(ln10)
site_map.add_child(ln11)
site_map.add_child(ln12)
site_map.add_child(ln13)
site_map.add_child(ln14)
site_map.add_child(ln15)
site_map.add_child(ln16)

site_map


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
